## Prerequisites

- A Python kernel that has `rpy2`, `pandas`, and `rpy-bridge` installed.
- R must be installed and available on `PATH` (rpy2 needs R).
- This repo should contain `examples/toy_funcs.R` (we added it to `examples/`).

If you are missing R or rpy2, install them before running these cells. In CI you must install R (we added `r-lib/actions/setup-r` to the workflow).

In [ ]:
# Environment checks: verify Python dependencies and R availability
import sys
from pathlib import Path

print("Python", sys.version.splitlines()[0])

try:
    import rpy2  # noqa: F401

    print("rpy2: OK")
except Exception as e:
    print("rpy2 import failed:", e)

# Check R command availability
import shutil, subprocess

R_cmd = shutil.which("R")
if R_cmd:
    try:
        out = subprocess.run([R_cmd, "--version"], capture_output=True, text=True, timeout=5)
        print("R available:", out.stdout.splitlines()[0])
    except Exception as e:
        print("R found but --version failed:", e)
else:
    print("R not found on PATH; rpy2 will not work until R is installed")

# Check example script path
script = Path("examples/toy_funcs.R")
print("example script exists:", script.exists(), script)

## Run the local example script

This uses the local `examples/toy_funcs.R` file. The R functions are simple and safe: `add_and_scale` and `multiply_table`.

In [ ]:
from pathlib import Path
from rpy_bridge import RFunctionCaller

script = Path("examples/toy_funcs.R")
caller = RFunctionCaller(path_to_renv=None, script_path=script)

print("add_and_scale(2,3) ->")
print(caller.call("add_and_scale", 2, 3))

print("add_and_scale(2,3, scale=10) ->")
print(caller.call("add_and_scale", 2, 3, scale=10))

print("multiply_table(2,5,times=4) -> pandas DataFrame")
df = caller.call("multiply_table", 2, 5, times=4)
print(type(df))
print(df.head())

## Using `renv`

If your R project uses `renv` to manage packages, pass the project directory (the parent that contains `renv/`) as `path_to_renv` when creating the caller. `RFunctionCaller` will call `renv::load()` before sourcing the script so the R session uses the project's library versions.

Only run the next cell if you have a local project with `renv` restored.

In [ ]:
# Example showing how to pass a local renv project (commented out by default)
from pathlib import Path

# local_project should be the folder that contains renv/activate.R and renv.lock
local_project = Path("/path/to/local/project")  # <-- change this if you have an renv
# script = local_project / 'examples' / 'toy_funcs.R'
# caller = RFunctionCaller(path_to_renv=local_project, script_path=script)
# print(caller.call('add_and_scale', 1, 2, scale=3))

print("Example shows how to supply path_to_renv; commented out to avoid accidental execution")

## Inspecting a script fetched from GitHub (safe flow)

Best practice:
1. Inspect the downloaded file first with `trust_remote_code=False`.
2. Review the file content.
3. If comfortable, run with `trust_remote_code=True`.

The downloaded file is cached under `~/.cache/rpy-bridge` keyed by repo+SHA. For private repos use `require_token=True` and set `GITHUB_TOKEN` in CI or be prepared to paste a token interactively in a TTY.

In [ ]:
from rpy_bridge import call_r_function_from_github

repo = "vic-cheung/rpy-bridge"  # change if needed
remote_path = "examples/toy_funcs.R"

# Inspect-only (safe): returns a Path to the cached downloaded file
cached = call_r_function_from_github(
    repo=repo,
    file_path=remote_path,
    function_name="add_and_scale",
    trust_remote_code=False,
)
print("Downloaded script cached at:", cached)

## Execute the remote script (opt-in)

Only run this after you have reviewed the script. `trust_remote_code=True` is required to construct an `RFunctionCaller` from the fetched script and execute it.

The example below is commented out; uncomment to run after inspection.

In [ ]:
# Uncomment and run after reviewing the fetched script above
# result = call_r_function_from_github(
#     repo=repo,
#     file_path=remote_path,
#     function_name='add_and_scale',
#     trust_remote_code=True,  # MUST be True to execute fetched code
#     path_to_renv=None,       # or provide a local project dir to activate renv
#     require_token=False,     # True for private repos
#     *[4, 6],                 # positional args -> x=4, y=6
#     scale=2,                 # named arg -> scale=2
# )
# print('Remote result:', result)

## How Python args map to R

- `caller.call('f', 1, 2)` -> R `f(1, 2)`
- `caller.call('f', a=1, b=2)` -> R `f(a = 1, b = 2)`
- Mix positional then named: `caller.call('f', 1, b=2)` -> `f(1, b = 2)`

Return mapping:
- R `data.frame` -> pandas `DataFrame`
- R named `list` -> Python `dict`
- Scalars -> Python `int/float`

## Next steps / exercises

- Modify `examples/toy_funcs.R` to return a summary `data.frame` and call it from Python.
- Try the GitHub inspect flow on a trusted repository.
- Add a small `renv` to a test project and pass `path_to_renv` to the caller.

If something fails (missing R, rpy2 import error), paste the error output here and I will help debug.